# Country-Level Longer-Term Forecasting of First-Time Asylum Applications

*A time-series analysis and forecasting exercise using Italy monthly Eurostat data*

This notebook uses monthly first-time asylum application data for Italy to demonstrate a longer-term country-level forecasting workflow.

The purpose is demonstrative and illustrative. The notebook is not an operational forecast product, and it does not attempt to explain the full political, legal, administrative, or structural dynamics behind asylum applications in Italy. Instead, it shows how a real administrative time series can be prepared, explored, split into training and test periods, modelled with transparent time-series methods, and interpreted with appropriate caution.

The target series is **monthly first-time asylum applications**. This brings in some caveats. Asylum applications are administrative events. They are not the same as regular or irregular entries, border arrivals, total displacement movements, or the number of people in need of protection at any given time. First-time applications may be affected by access to the asylum procedure, border management, administrative capacity, policy changes, onward movement, and wider dynamics in a country's asylum system or in countries of origin. They may also be affected by changes in the legal circumstances of certain individuals or groups of people.

For that reason, the forecasts in this notebook should be read as model-based projections of an administrative series, not as direct predictions of future arrivals or the volume of international protection needs.

# Part I - Understanding the time series

## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


## 2. Load prepared monthly series

The public notebook uses a cleaned monthly CSV derived from a locally downloaded Eurostat Excel file.

The original Excel file had countries as rows and monthly values as columns, with monthly date columns alternating with status or blank columns. During preparation, the Italy row was selected, monthly columns were parsed, Eurostat missing markers were converted, and incomplete months were excluded.

The modelling series used here runs from **January 2008 to February 2026**. The incomplete months **March 2026** and **April 2026** are not imputed and are not included.


In [ ]:
data_path_candidates = [
    Path("data/processed/italy_first_time_asylum_monthly.csv"),
    Path("../../data/processed/italy_first_time_asylum_monthly.csv"),
]
data_path = next(path for path in data_path_candidates if path.exists())

applications = (
    pd.read_csv(data_path, parse_dates=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

applications_display = applications.rename(
    columns={"date": "Month", "applications": "First-time applications"}
)
applications_display.head()


In [ ]:
applications_display.tail()


Before plotting the series, we check the basic structure of the data.


In [ ]:
summary_table = pd.DataFrame(
    {
        "Measure": [
            "Start date",
            "End date",
            "Number of months",
            "Total applications",
            "Mean monthly applications",
            "Median monthly applications",
            "Minimum monthly applications",
            "Maximum monthly applications",
            "Missing values",
        ],
        "Value": [
            applications["date"].min().strftime("%B %Y"),
            applications["date"].max().strftime("%B %Y"),
            len(applications),
            f"{applications['applications'].sum():,.0f}",
            f"{applications['applications'].mean():,.1f}",
            f"{applications['applications'].median():,.1f}",
            f"{applications['applications'].min():,.0f}",
            f"{applications['applications'].max():,.0f}",
            int(applications.isna().sum().sum()),
        ],
    }
)

summary_table


The series contains monthly counts of first-time asylum applications in Italy. The values are non-negative integer counts. The date range and missing-value checks confirm that the processed file is ready for exploratory time-series analysis.

## 3. Exploratory time-series analysis

We start by plotting the raw monthly series.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications["date"],
    applications["applications"],
    linewidth=1.2,
)
ax.set_title("Italy monthly first-time asylum applications")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.25)
fig.tight_layout()


The raw series shows substantial variation over time. Some periods have relatively low monthly application counts, while others show much higher pressure. This is expected for an asylum application series: the observed counts can respond to conflict dynamics, route dynamics, access to territory and procedures, policy changes, registration practices, and administrative capacity.

The raw monthly pattern is useful, but it is also noisy. To make the longer movement easier to see, we add a 12-month rolling average.


In [ ]:
applications_with_rolling = applications.assign(
    rolling_12_month_average=applications["applications"].rolling(12).mean()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["applications"],
    linewidth=0.9,
    alpha=0.45,
    label="Monthly applications",
)
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["rolling_12_month_average"],
    linewidth=2.0,
    label="12-month rolling average",
)
ax.set_title("Italy first-time asylum applications with 12-month rolling average")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


The 12-month rolling average smooths short-term month-to-month variation and makes broader phases more visible. It does not remove the underlying uncertainty or explain the causes of change. It simply helps separate longer movements from monthly fluctuation.

For an annual view, we aggregate the monthly counts by calendar year.


In [ ]:
annual_applications = (
    applications.assign(year=applications["date"].dt.year)
    .groupby("year", as_index=False)["applications"]
    .sum()
    .rename(columns={"year": "Year", "applications": "Total applications"})
)

annual_applications


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    annual_applications["Year"].astype(str),
    annual_applications["Total applications"],
)
ax.set_title("Annual first-time asylum applications in Italy")
ax.set_xlabel("Year")
ax.set_ylabel("Applications")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()


The annual summary gives a clearer sense of scale. It also shows why a simple average is unlikely to be enough for forecasting. The series includes quieter years, higher-pressure years, and periods where the level changes substantially.

At this stage, the main lesson is descriptive: the series is not stable around one constant level. This is why the rest of the notebook moves from exploration to model-based forecasting. We will first compare simple baseline forecasts, then fit SARIMA models that account for time dependence and seasonality, and finally test whether a SARIMAX model with descriptive exogenous event-period indicators adds value.

Before doing that, we need an evaluation design. In forecasting, it is not enough to fit a model to all available observations and inspect how well it describes the past. We need to ask whether the model can make useful predictions for observations it has not seen. The next part therefore defines a training period and a test period. The training period will be used to inspect the time-series structure and fit the models; the test period will be held back so that the baselines, SARIMA models, and SARIMAX model can be evaluated against later observations.
